# Azure Auto ML for Tabular Data

## Notebook Setup

In [7]:
import os
# Here we set the working directory to the project root to ensure imports work correctly
from pathlib import Path, os
target = "dp100-learn"
p = Path.cwd()
print(f"Starting working directory: {p}")
while p.name != target and p.parent != p:
    p = p.parent
p = "/mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn"
# p = "C:/Users/dmika/DEV/Projects-local/dp100-learn"
os.chdir(p)
print("Changed working directory to:", p)
from utils.azureml_utils import *

# Get Azure ML Client based on your environment. Learn more in the tutorials/azureml-first-notebook.ipynb.
ml_client = get_azureml_client()

Found the config file in: /config.json


In [2]:
# Setup experiment name
experiment_name = "dmdp100-automl-tabular-classification"

## Prepare the data
To pass a dataset as an input to an automated machine learning job, the data must be in tabular form and include a target column. For the data to be interpreted as a tabular dataset, the input dataset must be a **MLTable**.

In [4]:
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml import Input

# creates a dataset based on the files in the local data folder
my_training_data_input = Input(type=AssetTypes.MLTABLE, path="azureml:diabetes-data-table:1")

## Configure automated machine learning job

In [10]:
from azure.ai.ml.automl import ClassificationPrimaryMetrics
 
list(ClassificationPrimaryMetrics)

[<ClassificationPrimaryMetrics.AUC_WEIGHTED: 'AUCWeighted'>,
 <ClassificationPrimaryMetrics.ACCURACY: 'Accuracy'>,
 <ClassificationPrimaryMetrics.NORM_MACRO_RECALL: 'NormMacroRecall'>,
 <ClassificationPrimaryMetrics.AVERAGE_PRECISION_SCORE_WEIGHTED: 'AveragePrecisionScoreWeighted'>,
 <ClassificationPrimaryMetrics.PRECISION_SCORE_WEIGHTED: 'PrecisionScoreWeighted'>]

In [ ]:
from azure.ai.ml import automl

# configure the classification job
classification_job = automl.classification(
    compute="dmdp100-cpu-cluster",
    experiment_name=experiment_name,
    display_name="diabetes-classification-automl-job",
    training_data=my_training_data_input,
    target_column_name="Diabetic",
    primary_metric="accuracy",
    n_cross_validations=5,
    enable_model_explainability=True
)

# set the limits (optional)
classification_job.set_limits(
    timeout_minutes=90, 
    trial_timeout_minutes=20, 
    max_trials=5,
    enable_early_termination=True,
)

# set the training properties (optional)
classification_job.set_training(
    blocked_training_algorithms=["LogisticRegression"], 
    enable_onnx_compatible_models=True
)

## Run an automated machine learning job

OK, you're ready to go. Let's run the automated machine learning experiment.

> **Note**: This may take some time!

In [12]:
# Submit the AutoML job
returned_job = ml_client.jobs.create_or_update(
    classification_job
)  

# submit the job to the backend
aml_url = returned_job.studio_url
print("Monitor your job at", aml_url)

Monitor your job at https://ml.azure.com/runs/shy_machine_z6rhrnnjr8?wsid=/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourcegroups/polandaidevml-rg/workspaces/polandaidevml-mlw&tid=f78eb1c3-c2e5-404f-bd9e-9f6158703475


## Data Featurization

In [25]:
from azure.ai.ml import automl

# configure the classification job
classification_job = automl.classification(
    compute="dmdp100-cpu-cluster",
    experiment_name=experiment_name,
    display_name="diabetes-classification-automl-job-featurization",
    training_data=my_training_data_input,
    target_column_name="Diabetic",
    primary_metric="auc_weighted",
    n_cross_validations=5,
    enable_model_explainability=True
)

# set the limits (optional)
classification_job.set_limits(
    timeout_minutes=90, 
    trial_timeout_minutes=20, 
    max_trials=5,
    enable_early_termination=True,
)

# set the training properties (optional)
classification_job.set_training(
    blocked_training_algorithms=["LogisticRegression"], 
    enable_onnx_compatible_models=True
)

In [16]:
import pandas as pd
import mltable

data_asset = ml_client.data.get("diabetes-data-table", version="1")
tbl = mltable.load(f"azureml:/{data_asset.id}")
df = tbl.to_pandas_dataframe()
df.head(5)

Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


,PatientID,Pregnancies,PlasmaGlucose,DiastolicBloodPressure,TricepsThickness,SerumInsulin,BMI,DiabetesPedigree,Age,Diabetic
0,1354778,0,171,80,34,23,43.509726,1.213191,21,False
1,1147438,8,92,93,47,36,21.240576,0.158365,23,False
2,1640031,7,115,47,52,35,41.511523,0.079019,23,False
3,1883350,9,103,78,25,304,29.582192,1.282870,43,True
4,1424119,1,85,59,27,35,42.604536,0.549542,22,False


In [22]:
df.isna().sum(axis=0)

PatientID                 0
Pregnancies               0
PlasmaGlucose             0
DiastolicBloodPressure    0
TricepsThickness          0
SerumInsulin              0
BMI                       0
DiabetesPedigree          0
Age                       0
Diabetic                  0
dtype: int64

In [30]:
from azure.ai.ml.automl import ColumnTransformer

transformer_params = {
    "imputer": [
        ColumnTransformer(fields=["BMI"], parameters={"strategy": "most_frequent"}),
    ],
}
classification_job.set_featurization(
    mode="custom",
    transformer_params=transformer_params,
    column_name_and_types={"Age": "Numeric"},
)

In [31]:
# Submit the AutoML job
returned_job = ml_client.jobs.create_or_update(
    classification_job
)  

# submit the job to the backend
aml_url = returned_job.studio_url
print("Monitor your job at", aml_url)

Monitor your job at https://ml.azure.com/runs/tender_basil_pz8bkjpvsl?wsid=/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourcegroups/polandaidevml-rg/workspaces/polandaidevml-mlw&tid=f78eb1c3-c2e5-404f-bd9e-9f6158703475
